In [5]:
import numpy as np
from matplotlib import pyplot as plt

## test the PDE integrator

Grid is repeatedly updated using the following PDE-like rule

In [6]:
# from tqdm import tqdm
# from scipy.ndimage import gaussian_filter
# from pathlib import Path
# import imageio.v2 as imageio

# def f_A_B(A0, B0, dt, param):
#     # input: A0, B0: 2D arrays of shape (nx, ny)
#     # output: A1, B1: 2D arrays of shape (nx, ny)
#     # A1 = A0 + dt * (param['D_A'] * Laplacian(A0) - A0 * B0**2 + param['f'] * (1 - A0))
#     # B1 = B0 + dt * (param['D_B'] * Laplacian(B0) + A0 * B0**2 - (param['k'] + param['f']) * B0)
#     A1 = A0 + dt * (param['D_A'] * laplacian(A0, param['PBC']) - A0 * B0**2 + param['f'] * (1 - A0))
#     B1 = B0 + dt * (param['D_B'] * laplacian(B0, param['PBC']) + A0 * B0**2 - (param['k'] + param['f']) * B0)

#     # # clip values to [0,1]
#     # A1 = np.clip(A1, 0, 1)
#     # B1 = np.clip(B1, 0, 1)
#     return A1, B1

# def laplacian(Z, PBC=False):
#     # compute the Laplacian of Z using finite difference with custom weights
#     if PBC:
#         # Periodic boundary conditions
#         Zxx = np.roll(Z, 1, axis=0) - 2 * Z + np.roll(Z, -1, axis=0)
#         Zyy = np.roll(Z, 1, axis=1) - 2 * Z + np.roll(Z, -1, axis=1)
#         return Zxx + Zyy
#     else:
#         # Non-periodic boundary conditions with 3x3 convolution
#         result = np.zeros_like(Z)
#         for i in range(1, Z.shape[0] - 1):
#             for j in range(1, Z.shape[1] - 1):
#                 result[i, j] = (-1 * Z[i, j] + 
#                                0.2 * (Z[i-1, j] + Z[i+1, j] + Z[i, j-1] + Z[i, j+1]) +
#                                0.05 * (Z[i-1, j-1] + Z[i-1, j+1] + Z[i+1, j-1] + Z[i+1, j+1]))
#         return result

# # define grid size
# nx, ny = 256, 256

# # parameters
# param = {
#     'D_A': 0.1,
#     'D_B': 0.05,
#     'f': 0.03264,
#     'k': 0.06100,
#     'PBC': True
# }

# # initial conditions
# A0 = np.ones((nx, ny))
# B0 = np.zeros((nx, ny))
# # Set B=1 where random values are greater than a threshold
# threshold = 0.6
# B0_smooth = gaussian_filter(np.random.rand(nx, ny), sigma=2)
# # B0 = B0_smooth
# B0[B0_smooth > threshold] = 1.0

# # time step
# dt = 0.5

# # run the simulation for a few steps and plot the results, store in gif
# frames_dir = Path("frames")
# frames_dir.mkdir(exist_ok=True)

# for t in tqdm(range(10000)):
#     A0, B0 = f_A_B(A0, B0, dt, param)
#     if t % 250 == 0:
#         fig, ax = plt.subplots(1, 1, figsize=(6, 6))
#         ax.imshow(A0, cmap="Reds")
#         ax.set_axis_off()
#         fig.savefig(frames_dir / f"frame_{t:06d}.png", bbox_inches="tight", pad_inches=0)
#         plt.close(fig)

# frame_files = sorted(frames_dir.glob("frame_*.png"))
# with imageio.get_writer("A0_evolution.gif", mode="I", duration=0.2, loop=0) as writer:
#     for file in frame_files:
#         writer.append_data(imageio.imread(file))

# print(f"Saved GIF with {len(frame_files)} frames to A0_evolution.gif")


In [ ]:
from tqdm import tqdm
from scipy.ndimage import gaussian_filter
from pathlib import Path
import imageio.v2 as imageio

def f_A_B(A0, B0, dt, param):
    f = param["f"]
    k = param["k"]
    clip = param.get("clip", False)

    A1 = A0 + dt * (
        param["D_A"] * laplacian(A0, param["PBC"])
        - A0 * B0**2
        + f * (1 - A0)
    )
    B1 = B0 + dt * (
        param["D_B"] * laplacian(B0, param["PBC"])
        + A0 * B0**2
        - (k + f) * B0
    )
    # if clip:
    #     A1 = np.clip(A1, 0, 1)
    #     B1 = np.clip(B1, 0, 1)

    return A1, B1


def laplacian(Z, PBC=False):
    if PBC:
        # Zxx = np.roll(Z, 1, axis=0) - 2 * Z + np.roll(Z, -1, axis=0)
        # Zyy = np.roll(Z, 1, axis=1) - 2 * Z + np.roll(Z, -1, axis=1)
        # return Zxx + Zyy
        return laplacian_4th(Z)
    else:
        result = np.zeros_like(Z)
        for i in range(1, Z.shape[0] - 1):
            for j in range(1, Z.shape[1] - 1):
                result[i, j] = (
                    -1 * Z[i, j]
                    + 0.2 * (Z[i - 1, j] + Z[i + 1, j] + Z[i, j - 1] + Z[i, j + 1])
                    + 0.05 * (Z[i - 1, j - 1] + Z[i - 1, j + 1] + Z[i + 1, j - 1] + Z[i + 1, j + 1])
                )
        return result

def laplacian_4th(Z, h=1.0):
    Zxx = (
        -np.roll(Z,  2, axis=0)
        + 16*np.roll(Z,  1, axis=0)
        - 30*Z
        + 16*np.roll(Z, -1, axis=0)
        - np.roll(Z, -2, axis=0)
    ) / (12*h**2)

    Zyy = (
        -np.roll(Z,  2, axis=1)
        + 16*np.roll(Z,  1, axis=1)
        - 30*Z
        + 16*np.roll(Z, -1, axis=1)
        - np.roll(Z, -2, axis=1)
    ) / (12*h**2)

    return Zxx + Zyy

# define grid size
nx, ny = 512, 512

# heterogeneous reaction parameters
f_low, f_high = 0.035, 0.1
k_low, k_high = 0.055, 0.07

f_x = np.linspace(f_low, f_high, nx)[:, None]
k_y = np.linspace(k_low, k_high, ny)[None, :]
f_x_shift = f_x - (0.5-np.linspace(0, 1, ny)[None, :])**2 * (f_high - f_low)

param = {
    'D_A': 0.08,
    'D_B': 0.04,
    'PBC': True
}
param = {
    **param,
    "f": np.broadcast_to(f_x, (nx, ny)).copy(),
    "k": np.broadcast_to(k_y, (nx, ny)).copy(),
    "clip": True
}

dt = 1

# initial conditions
A0 = np.ones((nx, ny))
B0 = np.zeros((nx, ny))
# Set B=1 where random values are greater than a threshold
threshold = 0.6
B0_smooth = gaussian_filter(np.random.rand(nx, ny), sigma=2)
B0[B0_smooth > threshold] = 1.0

# run the simulation and store frames
frames_dir = Path("frames")
frames_dir.mkdir(exist_ok=True)

for file in frames_dir.glob("frame_*.png"):
    file.unlink()

for t in tqdm(range(10001)):
    A0, B0 = f_A_B(A0, B0, dt, param)

    if t % 100 == 0:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(A0, cmap="rainbow")
        ax.set_axis_off()
        fig.savefig(frames_dir / f"frame_{t:06d}.png", bbox_inches="tight", pad_inches=0)
        plt.close(fig)

frame_files = sorted(frames_dir.glob("frame_*.png"))
with imageio.get_writer("A0_evolution.gif", mode="I", duration=0.2, loop=0) as writer:
    for file in frame_files:
        writer.append_data(imageio.imread(file))

print(f"Saved GIF with {len(frame_files)} frames to A0_evolution.gif")

100%|██████████| 10001/10001 [10:44<00:00, 15.51it/s]


Saved GIF with 101 frames to A0_evolution.gif


In [8]:
frame_files = sorted(frames_dir.glob("frame_*.png"))
with imageio.get_writer("A0_evolution.gif", mode="I", duration=0.2, loop=0) as writer:
    for file in frame_files:
        writer.append_data(imageio.imread(file))

print(f"Saved GIF with {len(frame_files)} frames to A0_evolution.gif")

Saved GIF with 101 frames to A0_evolution.gif
